1. Imports

In [0]:
from pyspark.sql.functions import col, to_date, to_timestamp, current_timestamp, unix_timestamp, round, from_json
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType

2. silver_cardiovascular_age

In [0]:
df = spark.table("workspace.oura.bronze_cardiovascular_age")

df_silver = df.select(
    col("id"),
    to_date(col("day")).alias("day"),
    col("pulse_wave_velocity").cast(DoubleType()),
    col("vascular_age").cast(IntegerType()),
    col("ingested_at")
)

df_silver.write.format("delta").mode("overwrite").saveAsTable("workspace.oura.silver_cardiovascular_age")
print(f"✓ silver_cardiovascular_age: {df_silver.count()} rows")

✓ silver_cardiovascular_age: 687 rows


3. silver_readiness (with contributors parsing)

In [0]:
contributors_schema = StructType([
    StructField("activity_balance", IntegerType()),
    StructField("body_temperature", IntegerType()),
    StructField("hrv_balance", IntegerType()),
    StructField("previous_day_activity", IntegerType()),
    StructField("previous_night", IntegerType()),
    StructField("recovery_index", IntegerType()),
    StructField("resting_heart_rate", IntegerType()),
    StructField("sleep_balance", IntegerType()),
    StructField("sleep_regularity", IntegerType())
])

df = spark.table("workspace.oura.bronze_readiness")

df_silver = df.withColumn("contrib", from_json(col("contributors"), contributors_schema)) \
    .select(
        col("id"),
        to_date(col("day")).alias("day"),
        col("score").cast(IntegerType()).alias("readiness_score"),
        col("temperature_deviation").cast(DoubleType()),
        col("temperature_trend_deviation").cast(DoubleType()),
        col("contrib.activity_balance"),
        col("contrib.body_temperature"),
        col("contrib.hrv_balance"),
        col("contrib.previous_day_activity"),
        col("contrib.previous_night"),
        col("contrib.recovery_index"),
        col("contrib.resting_heart_rate"),
        col("contrib.sleep_balance"),
        col("contrib.sleep_regularity"),
        col("ingested_at")
    )

df_silver.write.format("delta").mode("overwrite").saveAsTable("workspace.oura.silver_readiness")
print(f"✓ silver_readiness: {df_silver.count()} rows")

✓ silver_readiness: 693 rows


4. silver_stress

In [0]:
df = spark.table("workspace.oura.bronze_stress")

df_silver = df.select(
    col("id"),
    to_date(col("day")).alias("day"),
    col("stress_high").cast(IntegerType()),
    col("recovery_high").cast(IntegerType()),
    col("ingested_at")
)

df_silver.write.format("delta").mode("overwrite").saveAsTable("workspace.oura.silver_stress")
print(f"✓ silver_stress: {df_silver.count()} rows")

✓ silver_stress: 724 rows


5. silver_heartrate

In [0]:
df = spark.table("workspace.oura.bronze_heartrate")

df_silver = df.select(
    to_timestamp(col("timestamp")).alias("timestamp"),
    col("bpm").cast(IntegerType()),
    col("source"),
    col("ingested_at")
)

df_silver.write.format("delta").mode("overwrite").saveAsTable("workspace.oura.silver_heartrate")
print(f"✓ silver_heartrate: {df_silver.count()} rows")

✓ silver_heartrate: 450141 rows


6. silver_workout

In [0]:
df = spark.table("workspace.oura.bronze_workout")

df_silver = df.select(
    col("id"),
    to_date(col("day")).alias("day"),
    col("activity"),
    col("calories").cast(DoubleType()),
    col("distance").cast(DoubleType()),
    col("intensity"),
    col("label"),
    col("source"),
    to_timestamp(col("start_datetime")).alias("start_datetime"),
    to_timestamp(col("end_datetime")).alias("end_datetime"),
    col("ingested_at")
).withColumn(
    "duration_minutes",
    round((unix_timestamp(col("end_datetime")) - unix_timestamp(col("start_datetime"))) / 60, 1)
)

df_silver.write.format("delta").mode("overwrite").saveAsTable("workspace.oura.silver_workout")
print(f"✓ silver_workout: {df_silver.count()} rows")

✓ silver_workout: 69 rows


In [0]:
%sql
SHOW TABLES IN workspace.oura

database,tableName,isTemporary
oura,bronze_cardiovascular_age,false
oura,bronze_heartrate,false
oura,bronze_readiness,false
oura,bronze_smoothed_cardiovascular_age,false
oura,bronze_stress,false
oura,bronze_workout,false
oura,silver_cardiovascular_age,false
oura,silver_heartrate,false
oura,silver_readiness,false
oura,silver_stress,false
